# Datamine CANON Process Example

This notebook demonstrates how to configure and run the **`canon`** process wrapper in `dmstudio`.

## Process Description

## CANON

Canonical analysis can be used to define possible mineralized and background samples by the calculation of canonical vectors and scores on the basis of two variates, left (mineralized) and right (background).

The number of fields for the left hand variate e.g.. mineralization, is specified by the user (@**NLEFT**). Obviously, it helps to have a pre-conceived idea of the data structure prior to analysis, i.e.. the number of fields to define for the left hand variate and the relevant fields to include as significant in defining the variates in the analysis.

Two variates are calculated from the linear correlation matrix (R) by the progressive selection of eigen values and eigen vectors. Once obtained, they are used to calculate the canonical vectors and canonical scores for each sample.

File Handling

Input data (&**IN**) has to contain a sample identifier (@**SAMPID**) which must be specified on input. The left hand variate fields must be the first fields occurring in the datafile &**IN**. If not, the data has to be sorted on the required left hand variate fields (keyfields) prior to analysis. Extra control can be applied by the selection of fields using field and/or retrieval criteria. The calculated correlation matrix (R), canonical vectors and significance tests are displayed and/or sent to a print file.

Canonical scores may be sent to an optional output file (&**SCORES**) which can be joined to the primary data using the **SAMPID** as a keyfield. The canonical scores with their associated grid coordinates can now be used for later data manipulation and map plotting.

**Note** : There is a restriction of 45 variables but there is no limit on the number of samples. If missing data values are present in the sample then the sample record is ignored. It is necessary to have the fields for the left hand variate defined as the first fields in the datafile.

### Input Files:

* **in** (Undefined):
  Input file.
  Required=Yes

### Output Files:

* **scores** (Undefined):
  Optional output file for canonical root scores.
  Required=No

### Fields:

* **sampid** (Undefined : IN):
  Field containing sample identification
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  First field to be used. No fields specified means all.
  Default=Undefined
  Required=No

### Parameters:

* **nleft**:
  Number of fields or variables in left hand part of the variate (1).
  Range=0,64
  Values=Undefined
  Default=1
  Required=No

* **print**:
  > 0 Display results on the screen (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('canon')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute canon
print("Running canon...")
dm_cmd.canon(
    in_i='t_assays',  # required
    scores_o=['t_canon_out'],  # required
    sampid_f='optional',  # required
    fields_f=['AU'],  # required
    # nleft_p=1,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("canon execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_canon_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")